# Tahap 1: Preprocessing & Feature Engineering
Notebook ini berfokus pada tiga tahapan utama untuk mempersiapkan data mentah Posyandu sebelum dilatih oleh Machine Learning:

1. **Pembersihan Data (Preprocessing):** Meratakan struktur data dari format Excel (*MultiIndex*) dan memfilter anomali nilai yang kosong.
2. **Rekayasa Fitur (Feature Engineering):** Mengekstrak fitur `Kecepatan_Tumbuh` (Tinggi dan Berat) secara dinamis. Ini memungkinkan model mendeteksi gejala awal gagal tumbuh (*Faltering Growth*) secara independen dari jumlah bulan pantauan.
3. **Penyeimbangan Data (Balancing & Augmentation):** Menyuntikkan profil klinis stunting buatan lalu menggunakan **SMOTE** untuk menghasilkan 142 baris dataset final. Tujuannya adalah menghilangkan bias kelas (*imbalanced data*) dan menambal kebutaan rentang umur pada data asli.

In [36]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data

In [37]:
raw_data_path = '../data/raw/dataset_posyandu.xlsx'
df = pd.read_excel(raw_data_path, header=[0, 1])

# Gabungkan multi-level header menjadi satu baris
new_cols = []
for c in df.columns:
    if 'Unnamed' in str(c[1]):
        new_cols.append(c[0])
    elif 'Unnamed' in str(c[0]):
        new_cols.append(c[1])
    else:
        new_cols.append(f"{c[0]}_{c[1]}")
df.columns = new_cols
df.head(3)

,No,Nama,Umur (Bulan),Jenis Kelamin,OrangTua,Alamat (Dusun),January_BB(kg),January_TB(cm),January_LL(cm),January_LK(cm),...,March_LK(cm),April_BB(kg),April_TB(cm),April_LL(cm),April_LK(cm),May_BB(kg),May_TB(cm),May_LL(cm),May_LK(cm),NORMAL(0) STUNTING (1)
0,1,CHAIRUNISA SALSABILA PUTRI,27,P,NaN,PAHING,10.8,81.7,16.0,45.0,...,46.0,10.80,84.0,14.0,48.0,11.2,86.0,15.0,49.0,0
1,2,DEANDRA ALFATHUNISA SYAFAZEA,36,P,NaN,PAHING,10.3,83.6,14.0,47.5,...,47.0,10.80,86.8,14.0,46.0,11.0,87.0,15.0,47.0,0
2,3,ARLEYAVA EVELYN PERMANA P,18,P,NaN,PAHING,8.0,72.0,14.0,42.0,...,43.0,8.67,74.3,14.0,43.0,9.0,75.0,44.0,15.0,0


### 2. Bersihkan Kolom

In [38]:
# Buang kolom identitas pasien
cols_to_drop = ['No', 'Nama', 'OrangTua', 'Alamat (Dusun)']
df_bersih = df.drop(columns=cols_to_drop, errors='ignore')

# Hapus baris kosong
df_bersih = df_bersih.dropna(how='all')
df_bersih.head(3)

,Umur (Bulan),Jenis Kelamin,January_BB(kg),January_TB(cm),January_LL(cm),January_LK(cm),February_BB(kg),February_TB(cm),February_LL(cm),February_LK(cm),...,March_LK(cm),April_BB(kg),April_TB(cm),April_LL(cm),April_LK(cm),May_BB(kg),May_TB(cm),May_LL(cm),May_LK(cm),NORMAL(0) STUNTING (1)
0,27,P,10.8,81.7,16.0,45.0,11.1,82.4,15.5,45.0,...,46.0,10.80,84.0,14.0,48.0,11.2,86.0,15.0,49.0,0
1,36,P,10.3,83.6,14.0,47.5,10.4,83.6,14.0,48.0,...,47.0,10.80,86.8,14.0,46.0,11.0,87.0,15.0,47.0,0
2,18,P,8.0,72.0,14.0,42.0,8.4,73.0,14.0,43.0,...,43.0,8.67,74.3,14.0,43.0,9.0,75.0,44.0,15.0,0


### 3. Feature Engineering Dinamis (Kecepatan Tumbuh)
Menciptakan fitur baru berdasarkan perhitungan matematis:
- **Kecepatan_Tumbuh_BB** = `(BB_Akhir - BB_Awal) / (Lama_Pantau - 1)`
- **Kecepatan_Tumbuh_TB** = `(TB_Akhir - TB_Awal) / (Lama_Pantau - 1)`
- **Rasio_BB_TB_Akhir** = `BB_Akhir / TB_Akhir`

Fitur ini berfungsi sebagai *Early Warning System* untuk mendeteksi *Faltering Growth* (Gagal Tumbuh) sebelum anak jatuh ke status Stunting.

In [39]:
bb_cols = [c for c in df_bersih.columns if 'BB' in c]
tb_cols = [c for c in df_bersih.columns if 'TB' in c]

# Ambil nilai awal dan akhir dari riwayat pengukuran
df_bersih['BB_Awal'] = df_bersih[bb_cols[0]]
df_bersih['TB_Awal'] = df_bersih[tb_cols[0]]
df_bersih['BB_Akhir'] = df_bersih[bb_cols[-1]]
df_bersih['TB_Akhir'] = df_bersih[tb_cols[-1]]

# Lama pantau dihitung dari jumlah kolom histori yang tersedia
df_bersih['Lama_Pantau_Bulan'] = len(bb_cols)

# Hitung kecepatan tumbuh per bulan (menghindari pembagian nol)
interval = df_bersih['Lama_Pantau_Bulan'] - 1
interval = interval.replace(0, 1) # Fallback jika hanya 1 bulan

df_bersih['Kecepatan_Tumbuh_BB'] = (df_bersih['BB_Akhir'] - df_bersih['BB_Awal']) / interval
df_bersih['Kecepatan_Tumbuh_TB'] = (df_bersih['TB_Akhir'] - df_bersih['TB_Awal']) / interval

df_bersih['Rasio_BB_TB_Akhir'] = df_bersih['BB_Akhir'] / (df_bersih['TB_Akhir'] + 0.001)

# Encoding Jenis Kelamin menjadi Numerik (L=1, P=0) agar bisa dibaca model ML
df_bersih['Jenis Kelamin'] = df_bersih['Jenis Kelamin'].map({'L': 1, 'P': 0})

# Seleksi hanya kolom fitur yang akan dipakai model
final_columns = ['Umur (Bulan)', 'Jenis Kelamin', 'BB_Awal', 'TB_Awal', 'BB_Akhir', 'TB_Akhir', 
                 'Lama_Pantau_Bulan', 'Kecepatan_Tumbuh_BB', 'Kecepatan_Tumbuh_TB', 'Rasio_BB_TB_Akhir', 'Status']

# Asumsi kolom target mengandung kata NORMAL atau STUNTING
target_col = [c for c in df_bersih.columns if 'NORMAL' in c.upper() or 'STUNTING' in c.upper()][0]
df_bersih = df_bersih.rename(columns={target_col: 'Status'})

df_fitur = df_bersih[final_columns]
df_fitur.head(3)

,Umur (Bulan),Jenis Kelamin,BB_Awal,TB_Awal,BB_Akhir,TB_Akhir,Lama_Pantau_Bulan,Kecepatan_Tumbuh_BB,Kecepatan_Tumbuh_TB,Rasio_BB_TB_Akhir,Status
0,27,0,10.8,81.7,11.2,86.0,5,0.100,1.075,0.130231,0
1,36,0,10.3,83.6,11.0,87.0,5,0.175,0.850,0.126435,0
2,18,0,8.0,72.0,9.0,75.0,5,0.250,0.750,0.119998,0


### 4. Injeksi Data Klinis Buatan

In [40]:
import numpy as np

# Hapus baris kosong jika ada
df_fitur = df_fitur.dropna()

# Kita buat 20 profil bayi stunting secara medis (umur 6-22 bulan)
np.random.seed(42)
num_inject = 20

df_injeksi = pd.DataFrame({
    'Umur (Bulan)': np.random.randint(6, 23, num_inject),
    'Jenis Kelamin': np.random.choice([0, 1], num_inject),
    'BB_Awal': np.round(np.random.uniform(5.0, 7.0, num_inject), 2),
    'TB_Awal': np.round(np.random.uniform(60.0, 71.0, num_inject), 2),
    'Lama_Pantau_Bulan': np.random.choice([3, 4, 5], num_inject),
    'Kecepatan_Tumbuh_BB': np.round(np.random.uniform(0.0, 0.05, num_inject), 3),
    'Kecepatan_Tumbuh_TB': np.round(np.random.uniform(0.0, 0.1, num_inject), 3),
    'Status': 1 # Label STUNTING
})

# Hitung nilai akhir berdasarkan rumus pasti
interval = df_injeksi['Lama_Pantau_Bulan'] - 1
df_injeksi['BB_Akhir'] = df_injeksi['BB_Awal'] + (df_injeksi['Kecepatan_Tumbuh_BB'] * interval)
df_injeksi['TB_Akhir'] = df_injeksi['TB_Awal'] + (df_injeksi['Kecepatan_Tumbuh_TB'] * interval)
df_injeksi['Rasio_BB_TB_Akhir'] = df_injeksi['BB_Akhir'] / df_injeksi['TB_Akhir']

# Gabungkan profil buatan ke dataset asli SEBELUM diserahkan ke SMOTE
df_fitur = pd.concat([df_fitur, df_injeksi], ignore_index=True)
print(f"Berhasil menyuntikkan {num_inject} pasien stunting buatan (bayi 6-22 bulan) ke dataset!")


Berhasil menyuntikkan 20 pasien stunting buatan (bayi 6-22 bulan) ke dataset!


### 5. Penyeimbangan Data Dinamis dengan SMOTE

In [41]:
from imblearn.over_sampling import SMOTE

X = df_fitur.drop(columns=['Status'])
y = df_fitur['Status']

print("Distribusi kelas SEBELUM SMOTE:")
print(y.value_counts())

# Inisialisasi SMOTE (Hanya akan menyamakan jumlah minoritas ke mayoritas)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Gabungkan hasil ke DataFrame final
df_final = pd.concat([X_resampled, y_resampled], axis=1)

# Acak urutan agar Normal dan Stunting bercampur
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nDistribusi kelas SESUDAH SMOTE:")
print(df_final['Status'].value_counts())
print(f"\nTotal Data Latih Siap Pakai: {len(df_final)} baris")


Distribusi kelas SEBELUM SMOTE:
Status
0    71
1    39
Name: count, dtype: int64

Distribusi kelas SESUDAH SMOTE:
Status
1    71
0    71
Name: count, dtype: int64

Total Data Latih Siap Pakai: 142 baris


### 5. Simpan Dataset

In [42]:
# Simpan ke folder processed
df_final.to_csv('../data/processed/dataset_final_training.csv', index=False)
print("Data training SMOTE berhasil disimpan ke 'data/processed/dataset_final_training.csv'")
df_final.head()


Data training SMOTE berhasil disimpan ke 'data/processed/dataset_final_training.csv'


,Umur (Bulan),Jenis Kelamin,BB_Awal,TB_Awal,BB_Akhir,TB_Akhir,Lama_Pantau_Bulan,Kecepatan_Tumbuh_BB,Kecepatan_Tumbuh_TB,Rasio_BB_TB_Akhir,Status
0,11,0,5.869504,64.680891,5.946443,64.833711,3,0.03847,0.07641,0.091906,1
1,49,1,15.000000,102.000000,15.000000,102.000000,5,0.00000,0.00000,0.147057,0
2,26,1,8.300000,79.100000,9.300000,79.100000,5,0.25000,0.00000,0.117571,1
3,16,1,6.370000,69.290000,6.442000,69.414000,3,0.03600,0.06200,0.092805,1
4,17,1,9.900000,73.800000,10.500000,75.500000,5,0.15000,0.42500,0.139071,0
